# Proposed Adaptive Attention heads Model

In [ ]:
!pip install wfdb

In [ ]:
# Importing Libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import h5py
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
from sklearn.utils import class_weight
from sklearn.model_selection import train_test_split
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import wfdb
import scipy.signal
from scipy import signal
from collections import Counter
import random
import json
import pandas as pd

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    # Model hyperparameters
    class_types = ['N', 'S', 'V', 'F', 'Q']
    num_heads = 8
    hidden_size = 2
    embedding = 16
    mlp_dim = 128
    kernel = 3
    emb_depth = 1
    transformer_layers = 1
    half_window = 99
    batch_size = 128

    # Training parameters
    learning_rate = 2e-3
    epochs = 100
    patience = 20
    random_seed = 42

    # Data paths and files
    splits_file = 'consistent_data_splits.npz'
    baseline_results = 'baseline_results.json'
    dynamic_weights = 'true_dynamic_ecg_transformer.weights.h5'
    dynamic_results = 'dynamic_results.json'

config = Config()

# Set all random seeds for reproducibility
np.random.seed(config.random_seed)
tf.random.set_seed(config.random_seed)
random.seed(config.random_seed)

In [ ]:
# =============================================================================
# COMMON MODEL COMPONENTS
# =============================================================================
def generate_patch_conv_orgPaper_f(embedding, inputs):
    patches = layers.Conv2D(filters=embedding, kernel_size=[config.kernel,1],
                           strides=[config.kernel,1], padding='valid')(inputs)
    for _ in range(config.emb_depth - 1):
        patches = layers.BatchNormalization()(patches)
        patches = layers.Conv2D(filters=embedding, kernel_size=[config.kernel,1],
                               strides=[config.kernel,1], padding='valid')(patches)

    row_axis, col_axis = (1, 2)
    seq_len = (patches.shape[row_axis]) * (patches.shape[col_axis])
    x = layers.Reshape((seq_len, embedding))(patches)
    return x

class AddPositionEmbs(layers.Layer):
    def __init__(self, posemb_init=None, **kwargs):
        super().__init__(**kwargs)
        self.posemb_init = posemb_init

    def build(self, inputs_shape):
        pos_emb_shape = (1, inputs_shape[1], inputs_shape[2])
        self.pos_embedding = self.add_weight(
            name='pos_embedding',
            shape=pos_emb_shape,
            initializer=self.posemb_init,
            trainable=True
        )

    def call(self, inputs, inputs_positions=None):
        pos_embedding = tf.cast(self.pos_embedding, inputs.dtype)
        return inputs + pos_embedding

def mlp_block_f(mlp_dim, inputs):
    x = layers.Dense(units=mlp_dim, activation=tf.nn.gelu)(inputs)
    x = layers.Dropout(rate=0.1)(x)
    x = layers.Dense(units=inputs.shape[-1], activation=tf.nn.gelu)(x)
    x = layers.Dropout(rate=0.1)(x)
    return x

In [ ]:
# =============================================================================
# ADAPTIVE DYNAMIC LAYER
# =============================================================================
class DynamicHeadGate(layers.Layer):
    def __init__(self, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads

        # Gating network - learns to predict head importance
        self.gate_net = tf.keras.Sequential([
            layers.Dense(32, activation='relu'),
            layers.Dropout(0.1),
            layers.Dense(16, activation='relu'),
            layers.Dropout(0.1),
            layers.Dense(num_heads)
        ])

        # Training step counter for temperature annealing
        self.training_step = tf.Variable(0, trainable=False, dtype=tf.int64, name="training_step")

    def build(self, input_shape):
        super().build(input_shape)

    def call(self, inputs, training=None):
        # Compute signal representation
        signal_repr = tf.reduce_mean(inputs, axis=1)  # [batch_size, embed_dim]

        # Generate head importance scores
        head_logits = self.gate_net(signal_repr, training=training)

        if training:
            # Update training step
            self.training_step.assign_add(1)

            # Temperature annealing - start high, gradually reduce
            global_step = tf.cast(self.training_step, tf.float32)
            temperature = tf.maximum(0.1, 2.0 - global_step / 10000.0)

            # Gumbel-Sigmoid for differentiable sampling
            uniform_noise = tf.random.uniform(tf.shape(head_logits), minval=1e-8, maxval=1.0)
            gumbel_noise = -tf.math.log(-tf.math.log(uniform_noise))
            gated_logits = (head_logits + gumbel_noise) / temperature
            head_probs = tf.nn.sigmoid(gated_logits)

            # regularization: encourage the model to be selective but not too sparse
            # This prevents using all heads or no heads - but doesn't force a specific number
            num_active = tf.reduce_sum(head_probs, axis=1)

            # Prevent complete shutdown and prevent using all heads
            # But let the model learn the right balance
            sparsity_penalty = tf.reduce_mean(
                tf.maximum(0.0, num_active - (self.num_heads * 0.8)) +  # Penalty for using >80%
                tf.maximum(0.0, 1.0 - num_active)  # Penalty for using <1 head
            )
            self.add_loss(0.1 * sparsity_penalty)

            return head_probs
        else:
            # Inference - hard thresholding based on learned importance
            head_probs = tf.nn.sigmoid(head_logits)

            # Simple threshold - let the model decide
            threshold = 0.9
            head_mask = tf.cast(head_probs > threshold, tf.float32)

            # Only ensure at least ONE head is active (prevent complete shutdown)
            num_active = tf.reduce_sum(head_mask, axis=1, keepdims=True)
            no_heads_active = tf.cast(num_active < 1.0, tf.float32)

            # If no heads active, use the best head
            def activate_best_head():
                _, best_head_idx = tf.nn.top_k(head_probs, k=1)
                batch_size = tf.shape(head_probs)[0]
                batch_indices = tf.expand_dims(tf.range(batch_size), 1)
                indices = tf.stack([batch_indices, best_head_idx], axis=-1)
                best_head_mask = tf.scatter_nd(
                    indices, tf.ones((batch_size, 1), dtype=tf.float32), tf.shape(head_probs)
                )
                return tf.where(no_heads_active > 0, best_head_mask, head_mask)

            def keep_current_mask():
                return head_mask

            final_mask = tf.cond(
                tf.reduce_any(no_heads_active > 0),
                activate_best_head,
                keep_current_mask
            )

            return final_mask

class DynamicMultiHeadAttention(layers.Layer):
    def __init__(self, num_heads, key_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.dropout_rate = dropout

        # Individual attention heads
        self.attention_heads = []
        for i in range(num_heads):
            self.attention_heads.append(
                layers.MultiHeadAttention(
                    num_heads=1,
                    key_dim=key_dim,
                    dropout=dropout,
                    name=f'head_{i}'
                )
            )

        # Output projection
        self.output_dense = layers.Dense(key_dim * num_heads)
        self.dropout_layer = layers.Dropout(dropout)

        # True dynamic gating
        self.head_gate = DynamicHeadGate(num_heads)

        # Statistics tracking
        self.head_usage_counts = tf.Variable(
            initial_value=tf.zeros((num_heads,), dtype=tf.int64),
            trainable=False, name="head_usage_counts"
        )
        self.total_forward_passes = tf.Variable(
            0, trainable=False, dtype=tf.int64, name="total_forward_passes"
        )
        self.total_heads_computed = tf.Variable(
            0, trainable=False, dtype=tf.int64, name="total_heads_computed"
        )
        self.heads_per_sample = tf.Variable(
            initial_value=tf.zeros((1000,), dtype=tf.float32),  # Store last 1000 samples
            trainable=False, name="heads_per_sample"
        )
        self.sample_idx = tf.Variable(0, trainable=False, dtype=tf.int64, name="sample_idx")

    def build(self, input_shape):
        super().build(input_shape)

    def compute_output_shape(self, input_shape):
        if isinstance(input_shape, list):
            return input_shape[0]
        return input_shape

    def call(self, query, value=None, key=None, training=None):
        if value is None:
            value = query
        if key is None:
            key = value

        batch_size = tf.shape(query)[0]

        # Get head activation mask - NO CONSTRAINTS, pure learning
        head_mask = self.head_gate(query, training=training)

        # Compute attention for each head
        head_outputs = []

        for i in range(self.num_heads):
            head_active = head_mask[:, i:i+1]
            head_output = self.attention_heads[i](query, value, key, training=training)

            # Apply gating
            head_active_expanded = tf.expand_dims(head_active, axis=-1)
            gated_output = head_output * head_active_expanded

            head_outputs.append(gated_output)

        # Track statistics
        def update_stats():
            self.total_forward_passes.assign_add(tf.cast(batch_size, tf.int64))
            batch_per_head_count = tf.cast(tf.reduce_sum(head_mask, axis=0), tf.int64)
            self.head_usage_counts.assign_add(batch_per_head_count)
            self.total_heads_computed.assign_add(tf.reduce_sum(batch_per_head_count))

            # Track heads per sample - simplified for graph compatibility
            heads_used = tf.reduce_sum(head_mask, axis=1)  # [batch_size]
            # Just update the counter, detailed tracking in eager mode only
            self.sample_idx.assign_add(tf.cast(batch_size, tf.int64))

            return tf.constant(0)

        def no_update():
            return tf.constant(0)

        # Only update stats in eager execution
        if tf.executing_eagerly():
            update_stats()

        # Concatenate and project
        concatenated = tf.concat(head_outputs, axis=-1)
        output = self.output_dense(concatenated)
        output = self.dropout_layer(output, training=training)

        return output

    def get_efficiency_stats(self):
        """Get comprehensive statistics showing model's learned behavior"""
        total_forward_passes = int(self.total_forward_passes.numpy())

        if total_forward_passes > 0:
            total_computed = int(self.total_heads_computed.numpy())
            total_possible = total_forward_passes * self.num_heads

            avg_heads = total_computed / total_forward_passes
            efficiency = 1.0 - (total_computed / total_possible)

            per_head_counts = [int(x) for x in self.head_usage_counts.numpy()]
            per_head_usage_rates = [(count / total_forward_passes) * 100.0
                                   for count in per_head_counts]

            # Analyze heads per sample distribution
            heads_per_sample = self.heads_per_sample.numpy()
            valid_samples = heads_per_sample[heads_per_sample > 0]

            if len(valid_samples) > 0:
                min_heads = float(np.min(valid_samples))
                max_heads = float(np.max(valid_samples))
                std_heads = float(np.std(valid_samples))

                # Distribution of head usage
                head_distribution = {}
                for i in range(self.num_heads + 1):
                    count = np.sum(np.abs(valid_samples - i) < 0.1)
                    head_distribution[f'{i}_heads'] = count
            else:
                min_heads = max_heads = std_heads = 0.0
                head_distribution = {}

            return {
                'avg_heads_used': avg_heads,
                'min_heads_used': min_heads,
                'max_heads_used': max_heads,
                'std_heads_used': std_heads,
                'total_heads': self.num_heads,
                'efficiency_gain': efficiency * 100.0,
                'total_computed': total_computed,
                'total_possible': total_possible,
                'total_forward_passes': total_forward_passes,
                'per_head_counts': per_head_counts,
                'per_head_usage_rates': per_head_usage_rates,
                'head_distribution': head_distribution
            }
        return None

    def reset_stats(self):
        """Reset all statistics"""
        self.total_heads_computed.assign(0)
        self.total_forward_passes.assign(0)
        self.head_usage_counts.assign(tf.zeros_like(self.head_usage_counts))
        self.heads_per_sample.assign(tf.zeros_like(self.heads_per_sample))
        self.sample_idx.assign(0)

def TrueDynamicEncoder1Dblock(num_heads, mlp_dim, inputs):
    """Pure dynamic transformer block"""
    x = layers.LayerNormalization(dtype=inputs.dtype)(inputs)
    x_att = DynamicMultiHeadAttention(
        num_heads=num_heads,
        key_dim=config.hidden_size,
        dropout=0.1
    )(x, x)
    x = layers.Add()([x_att, inputs])

    # MLP block
    y = layers.LayerNormalization(dtype=x.dtype)(x)
    y = mlp_block_f(mlp_dim, y)
    y_1 = layers.Add()([y, x])

    return y_1

def build_true_dynamic_ViT():
    """Build truly dynamic Vision Transformer"""
    inputs = layers.Input(shape=(198,1,1), name='x1')
    RR_feat = layers.Input(shape=(2,), name='x2')

    # Patch embedding
    patches = generate_patch_conv_orgPaper_f(config.embedding, inputs)
    x = AddPositionEmbs(
        posemb_init=tf.keras.initializers.RandomNormal(stddev=0.02),
        name='posembed_input'
    )(patches)
    x = layers.Dropout(rate=0.1)(x)

    #  Dynamic transformer block - learns from scratch
    x = TrueDynamicEncoder1Dblock(config.num_heads, config.mlp_dim, x)

    # Final processing
    encoded = layers.LayerNormalization(name='encoder_norm')(x)
    im_representation = layers.GlobalAveragePooling1D()(encoded)

    # RR interval features
    emb_RR = layers.Dense(units=4, activation='relu', use_bias=True)(RR_feat)
    emb_RR = layers.Dropout(0.1)(emb_RR)

    # Combine features
    combined = layers.Concatenate()([im_representation, emb_RR])
    combined = layers.Dense(32, activation='relu')(combined)
    combined = layers.Dropout(0.2)(combined)

    # Classification head
    logits = layers.Dense(
        units=len(config.class_types),
        name='head',
        kernel_initializer=tf.keras.initializers.TruncatedNormal(stddev=0.02)
    )(combined)

    model = tf.keras.Model(inputs=[inputs, RR_feat], outputs=logits)
    return model

In [ ]:
# =============================================================================
# DATASET LOADING
# =============================================================================
def load_data_splits():
    """Load existing data splits"""
    if not os.path.exists(config.splits_file):
        raise FileNotFoundError(
            f"Data splits file '{config.splits_file}' not found!\n"
            "Please run the baseline model script first to create the data splits."
        )

    print(f"Loading data splits from {config.splits_file}")
    data = np.load(config.splits_file)
    return (data['X_train'], data['X_val'], data['X_test'],
            data['y_train'], data['y_val'], data['y_test'],
            data['RR_train'], data['RR_val'], data['RR_test'])

def create_datasets(X_train, RR_train, y_train, X_val, RR_val, y_val, batch_size):
    """Create TensorFlow datasets for training"""
    autotune = tf.data.AUTOTUNE

    # Reshape for Conv2D
    X_train = X_train.reshape(-1, 198, 1, 1)
    X_val = X_val.reshape(-1, 198, 1, 1)

    # Convert to categorical
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=len(config.class_types))
    y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes=len(config.class_types))

    # Training dataset
    train_data = tf.data.Dataset.from_tensor_slices(X_train)
    train_RR = tf.data.Dataset.from_tensor_slices(RR_train)
    train_labels = tf.data.Dataset.from_tensor_slices(y_train_cat)

    X_train_ds = tf.data.Dataset.zip((train_data, train_RR)).map(
        lambda x1, x2: {'x1': x1, 'x2': x2}
    )
    train_ds = tf.data.Dataset.zip((X_train_ds, train_labels))
    train_ds = (train_ds.shuffle(10000, seed=config.random_seed)
                .batch(batch_size)
                .prefetch(autotune))

    # Validation dataset
    val_data = tf.data.Dataset.from_tensor_slices(X_val)
    val_RR = tf.data.Dataset.from_tensor_slices(RR_val)
    val_labels = tf.data.Dataset.from_tensor_slices(y_val_cat)

    X_val_ds = tf.data.Dataset.zip((val_data, val_RR)).map(
        lambda x1, x2: {'x1': x1, 'x2': x2}
    )
    val_ds = tf.data.Dataset.zip((X_val_ds, val_labels))
    val_ds = (val_ds.batch(batch_size)
              .prefetch(autotune))

    return train_ds, val_ds

def create_test_dataset(X_test, RR_test, y_test, batch_size):
    """Create test dataset"""
    X_test = X_test.reshape(-1, 198, 1, 1)
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes=len(config.class_types))

    test_data = tf.data.Dataset.from_tensor_slices(X_test)
    test_RR = tf.data.Dataset.from_tensor_slices(RR_test)
    test_labels = tf.data.Dataset.from_tensor_slices(y_test_cat)

    X_test_ds = tf.data.Dataset.zip((test_data, test_RR)).map(
        lambda x1, x2: {'x1': x1, 'x2': x2}
    )
    test_ds = tf.data.Dataset.zip((X_test_ds, test_labels))
    test_ds = test_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return test_ds

In [ ]:
# =============================================================================
# TRAINING FUNCTION
# =============================================================================
def train_true_dynamic_model(model, train_ds, val_ds, weights_path):
    """Train with dynamic attention from the very beginning"""
    print(f"\n🚀 Training True Dynamic Model with Pure Dynamic Attention...")
    print("No progressive training - learning optimal head usage from scratch!")

    # Compile with class weights for balanced learning
    optimizer = tf.keras.optimizers.Adam(learning_rate=config.learning_rate)
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'],
        run_eagerly=False
    )

    # Callbacks
    checkpoint = ModelCheckpoint(
        weights_path,
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=True,
        mode='max',
        verbose=1
    )

    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        patience=config.patience,
        restore_best_weights=True,
        mode='max',
        verbose=1
    )

    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-7,
        verbose=1
    )

    # Train with dynamic attention from epoch 1
    history = model.fit(
        train_ds,
        epochs=config.epochs,
        validation_data=val_ds,
        callbacks=[checkpoint, early_stopping, reduce_lr],
        verbose=1
    )

    print(f"✅ True Dynamic model training completed!")
    print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")

    return history

In [ ]:
# =============================================================================
# EVALUATION FUNCTION
# =============================================================================
def evaluate_true_dynamic_model(model, test_ds, y_test):
    """Enhanced evaluation showing what the model actually learned"""
    print(f"\n Evaluating True Dynamic Model...")

    # Reset statistics
    for layer in model.layers:
        if hasattr(layer, 'layers'):
            for sublayer in layer.layers:
                if isinstance(sublayer, DynamicMultiHeadAttention):
                    sublayer.reset_stats()
        elif isinstance(layer, DynamicMultiHeadAttention):
            layer.reset_stats()

    # Enable eager execution for statistics
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config.learning_rate),
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'],
        run_eagerly=True
    )

    # Collect predictions
    logits_list = []
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

    print("Running inference and analyzing head usage...")
    for batch_idx, batch in enumerate(test_ds):
        inputs_dict, labels_onehot = batch

        # Forward pass
        logits = model(inputs_dict, training=False)
        logits_np = logits.numpy()
        logits_list.append(logits_np)

        # Calculate metrics
        bsize = logits_np.shape[0]
        total_samples += bsize

        batch_loss = loss_fn(labels_onehot, logits).numpy()
        total_loss += batch_loss * bsize

        preds_int = np.argmax(logits_np, axis=1)
        true_int = np.argmax(labels_onehot.numpy(), axis=1)
        total_correct += np.sum(preds_int == true_int)

        if (batch_idx + 1) % 10 == 0:
            print(f"  Processed {batch_idx + 1} batches...")

    # Final metrics
    test_loss = total_loss / total_samples
    test_accuracy = total_correct / total_samples

    # Standard evaluation metrics
    all_logits = np.concatenate(logits_list, axis=0)
    y_pred_probs = tf.nn.softmax(all_logits, axis=1).numpy()
    y_pred = np.argmax(y_pred_probs, axis=1)

    # Detailed metrics with zero_division parameter
    precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

    precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f"\n📊 True Dynamic Model Test Results:")
    print(f"Test Loss:           {test_loss:.4f}")
    print(f"Test Accuracy:       {test_accuracy:.4f}")
    print(f"Precision (macro):   {precision_macro:.4f}")
    print(f"Recall (macro):      {recall_macro:.4f}")
    print(f"F1-Score (macro):    {f1_macro:.4f}")
    print(f"Precision (weighted): {precision_weighted:.4f}")
    print(f"Recall (weighted):   {recall_weighted:.4f}")
    print(f"F1-Score (weighted): {f1_weighted:.4f}")

    print(f"\n📋 Classification Report:")
    print(classification_report(y_test, y_pred, target_names=config.class_types, digits=4))

    # Analyze what the model learned
    attention_stats = {}
    for layer in model.layers:
        if hasattr(layer, 'layers'):
            for sublayer in layer.layers:
                if isinstance(sublayer, DynamicMultiHeadAttention):
                    stats = sublayer.get_efficiency_stats()
                    if stats:
                        attention_stats['dynamic_attention'] = stats

                        print(f"\n🧠 LEARNED HEAD USAGE PATTERNS:")
                        print(f"  Average heads per sample: {stats['avg_heads_used']:.2f}")
                        print(f"  Minimum heads used: {stats['min_heads_used']:.1f}")
                        print(f"  Maximum heads used: {stats['max_heads_used']:.1f}")
                        print(f"  Standard deviation: {stats['std_heads_used']:.2f}")
                        print(f"  Efficiency gain: {stats['efficiency_gain']:.1f}%")

                        print(f"\n📊 HEAD USAGE DISTRIBUTION:")
                        for k, v in stats['head_distribution'].items():
                            if v > 0:
                                print(f"  {k}: {v} samples")

                        print(f"\n🔍 PER-HEAD USAGE RATES:")
                        for i, rate in enumerate(stats['per_head_usage_rates']):
                            print(f"  Head {i}: {rate:.1f}%")

    return {
        'model_name': 'True Dynamic',
        'test_accuracy': float(test_accuracy),
        'test_loss': float(test_loss),
        'precision_macro': float(precision_macro),
        'recall_macro': float(recall_macro),
        'f1_macro': float(f1_macro),
        'precision_weighted': float(precision_weighted),
        'recall_weighted': float(recall_weighted),
        'f1_weighted': float(f1_weighted),
        'predictions': y_pred.tolist(),
        'true_labels': y_test.tolist(),
        'attention_stats': attention_stats
    }

In [ ]:
# =============================================================================
# VISUALIZATION FUNCTIONS
# =============================================================================
def plot_dynamic_training_history(history, save_path='dynamic_training_history.png'):
    """Plot dynamic training history"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # Loss plots
    axes[0, 0].plot(history.history['loss'], label='Train Loss')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    axes[0, 1].plot(history.history['val_loss'], label='Val Loss', color='orange')
    axes[0, 1].set_title('Validation Loss')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    # Accuracy plots
    axes[1, 0].plot(history.history['accuracy'], label='Train Accuracy')
    axes[1, 0].set_title('Training Accuracy')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    axes[1, 1].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
    axes[1, 1].set_title('Validation Accuracy')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Accuracy')
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_dynamic_confusion_matrix(results, save_path='dynamic_confusion_matrix.png'):
    """Plot confusion matrix for dynamic model"""
    y_true = results['true_labels']
    y_pred = results['predictions']

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
               xticklabels=config.class_types, yticklabels=config.class_types)
    plt.title(f'True Dynamic Model Confusion Matrix\nAccuracy: {results["test_accuracy"]:.4f}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_head_usage_analysis(attention_stats, save_path='head_usage_analysis.png'):
    """Plot detailed head usage analysis for dynamic model"""
    if not attention_stats:
        print("No attention statistics available for plotting.")
        return

    stats = list(attention_stats.values())[0]

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Head usage rates
    heads = list(range(len(stats['per_head_usage_rates'])))
    rates = stats['per_head_usage_rates']

    axes[0, 0].bar(heads, rates, alpha=0.7, color='skyblue')
    axes[0, 0].set_title('Per-Head Usage Rates')
    axes[0, 0].set_xlabel('Head Index')
    axes[0, 0].set_ylabel('Usage Rate (%)')
    axes[0, 0].grid(True, alpha=0.3)

    # Head distribution
    distribution = stats['head_distribution']
    if distribution:
        heads_count = []
        samples_count = []
        for k, v in distribution.items():
            if v > 0:
                heads_count.append(k.replace('_heads', ''))
                samples_count.append(v)

        axes[0, 1].bar(heads_count, samples_count, alpha=0.7, color='lightgreen')
        axes[0, 1].set_title('Distribution of Heads per Sample')
        axes[0, 1].set_xlabel('Number of Heads Used')
        axes[0, 1].set_ylabel('Number of Samples')
        axes[0, 1].grid(True, alpha=0.3)

    # Efficiency metrics
    metrics = ['avg_heads_used', 'efficiency_gain']
    values = [stats['avg_heads_used'], stats['efficiency_gain']]
    labels = ['Avg Heads Used', 'Efficiency Gain (%)']

    axes[1, 0].bar(labels, values, alpha=0.7, color=['orange', 'red'])
    axes[1, 0].set_title('Efficiency Metrics')
    axes[1, 0].set_ylabel('Value')
    axes[1, 0].grid(True, alpha=0.3)

    # Summary statistics
    summary_text = f"""
    Dynamic Attention Summary:

    Average heads used: {stats['avg_heads_used']:.2f} / {stats['total_heads']}
    Min heads used: {stats['min_heads_used']:.1f}
    Max heads used: {stats['max_heads_used']:.1f}
    Std deviation: {stats['std_heads_used']:.2f}

    Efficiency gain: {stats['efficiency_gain']:.1f}%
    Total forward passes: {stats['total_forward_passes']:,}
    """

    axes[1, 1].text(0.1, 0.5, summary_text, transform=axes[1, 1].transAxes,
                    fontsize=12, verticalalignment='center',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray"))
    axes[1, 1].set_xlim(0, 1)
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].axis('off')
    axes[1, 1].set_title('Summary Statistics')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def plot_model_comparison(baseline_results, dynamic_results, save_path='model_comparison.png'):
    """Plot comparison between baseline and dynamic models"""
    # Load baseline results if path provided
    if isinstance(baseline_results, str):
        with open(baseline_results, 'r') as f:
            baseline_results = json.load(f)

    metrics = ['test_accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
    baseline_values = [baseline_results[metric] for metric in metrics]
    dynamic_values = [dynamic_results[metric] for metric in metrics]

    fig, ax = plt.subplots(figsize=(12, 6))

    x = np.arange(len(metrics))
    width = 0.35

    ax.bar(x - width/2, baseline_values, width, label='Baseline', alpha=0.8, color='skyblue')
    ax.bar(x + width/2, dynamic_values, width, label='True Dynamic', alpha=0.8, color='lightgreen')

    # Add value labels on bars
    for i, (b_val, d_val) in enumerate(zip(baseline_values, dynamic_values)):
        ax.text(x[i] - width/2, b_val + 0.01, f'{b_val:.3f}',
               ha='center', va='bottom', fontweight='bold')
        ax.text(x[i] + width/2, d_val + 0.01, f'{d_val:.3f}',
               ha='center', va='bottom', fontweight='bold')

    ax.set_xlabel('Metrics')
    ax.set_ylabel('Score')
    ax.set_title('Baseline vs True Dynamic Model Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1-Score'])
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.1)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# =============================================================================
# MAIN EXECUTION
# =============================================================================
def main():
    """Main execution for true dynamic model"""
    print(" TRUE DYNAMIC ECG Transformer - Training & Evaluation")
    print("=" * 60)

    # Setup GPU if available
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f" Found {len(gpus)} GPU(s), memory growth enabled")
        except RuntimeError as e:
            print(f"GPU setup error: {e}")
    else:
        print("🔄 No GPU found, running on CPU")

    # Load existing data splits
    try:
        X_train, X_val, X_test, y_train, y_val, y_test, RR_train, RR_val, RR_test = load_data_splits()
    except FileNotFoundError as e:
        print(f"❌ {e}")
        return

    # Calculate class weights
    class_weights = class_weight.compute_class_weight(
        'balanced', classes=np.unique(y_train), y=y_train
    )
    class_weight_dict = dict(enumerate(class_weights))
    print(f"Class weights: {class_weight_dict}")

    # Create datasets
    train_ds, val_ds = create_datasets(X_train, RR_train, y_train, X_val, RR_val, y_val, config.batch_size)
    test_ds = create_test_dataset(X_test, RR_test, y_test, config.batch_size)

    # Build and train true dynamic model
    print("\n🏗️  Building True Dynamic Model...")
    dynamic_model = build_true_dynamic_ViT()
    print(f"True Dynamic model parameters: {dynamic_model.count_params():,}")

    # Check if model already exists
    if os.path.exists(config.dynamic_weights):
        print(f"\n Found existing dynamic weights: {config.dynamic_weights}")
        choice = input("Do you want to (1) Retrain, (2) Load existing, or (3) Skip training? [1/2/3]: ")

        if choice == '2':
            print(" Loading existing dynamic weights...")
            dynamic_model.load_weights(config.dynamic_weights)
            history = None
        elif choice == '3':
            print("⏭  Skipping dynamic training...")
            return
        else:
            print(" Retraining dynamic model...")
            history = train_true_dynamic_model(dynamic_model, train_ds, val_ds, config.dynamic_weights)
            dynamic_model.load_weights(config.dynamic_weights)
    else:
        history = train_true_dynamic_model(dynamic_model, train_ds, val_ds, config.dynamic_weights)
        dynamic_model.load_weights(config.dynamic_weights)

    # Evaluate dynamic model
    print("\n Evaluating True Dynamic Model...")
    results = evaluate_true_dynamic_model(dynamic_model, test_ds, y_test)

    # Save results
    with open(config.dynamic_results, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"💾 Results saved to {config.dynamic_results}")

    # Generate visualizations
    if history is not None:
        print("\n Generating training history plot...")
        plot_dynamic_training_history(history)

    print(" Generating confusion matrix...")
    plot_dynamic_confusion_matrix(results)

    # Plot head usage analysis if available
    if results['attention_stats']:
        print(" Generating head usage analysis...")
        plot_head_usage_analysis(results['attention_stats'])

    # Compare with baseline if available
    if os.path.exists(config.baseline_results):
        print(" Generating model comparison...")
        plot_model_comparison(config.baseline_results, results)

    # Final summary and analysis
    print("\n" + "="*60)
    print("🎉 TRUE DYNAMIC MODEL SUMMARY")
    print("="*60)
    print(f"Test Accuracy:     {results['test_accuracy']:.4f}")
    print(f"Test Loss:         {results['test_loss']:.4f}")
    print(f"F1-Score (macro):  {results['f1_macro']:.4f}")
    print(f"F1-Score (weighted): {results['f1_weighted']:.4f}")

    # What did the model actually learn?
    if results['attention_stats']:
        stats = list(results['attention_stats'].values())[0]
        print(f"\n WHAT THE MODEL LEARNED:")
        print(f"  The model chose to use {stats['avg_heads_used']:.1f} heads on average")
        print(f"  Range: {stats['min_heads_used']:.0f} - {stats['max_heads_used']:.0f} heads per sample")
        print(f"  This achieved {stats['efficiency_gain']:.1f}% computational savings")

        # Compare with baseline if available
        if os.path.exists(config.baseline_results):
            with open(config.baseline_results, 'r') as f:
                baseline_results = json.load(f)

            acc_diff = results['test_accuracy'] - baseline_results['test_accuracy']
            f1_diff = results['f1_macro'] - baseline_results['f1_macro']

            print(f"\n COMPARISON WITH BASELINE:")
            print(f"  Accuracy difference: {acc_diff:+.4f}")
            print(f"  F1-Score difference: {f1_diff:+.4f}")


    print(f"\n Files Generated:")
    print(f"  - {config.dynamic_weights} (model weights)")
    print(f"  - {config.dynamic_results} (detailed results)")
    print(f"  - dynamic_training_history.png")
    print(f"  - dynamic_confusion_matrix.png")
    print(f"  - head_usage_analysis.png")
    if os.path.exists(config.baseline_results):
        print(f"  - model_comparison.png")

    print(f"\n✅ True Dynamic model training and evaluation completed!")

    return results

if __name__ == "__main__":
    # Run dynamic pipeline
    results = main()

🚀 TRUE DYNAMIC ECG Transformer - Training & Evaluation
✅ Found 1 GPU(s), memory growth enabled
Loading data splits from consistent_data_splits.npz
Class weights: {0: np.float64(0.22354713868798376), 1: np.float64(7.246375321336761), 2: np.float64(2.872849571952711), 3: np.float64(25.123351158645278), 4: np.float64(1281.290909090909)}

🏗️  Building True Dynamic Model...
True Dynamic model parameters: 6,305

🔄 Found existing dynamic weights: true_dynamic_ecg_transformer.weights.h5
Do you want to (1) Retrain, (2) Load existing, or (3) Skip training? [1/2/3]: 3
⏭️  Skipping dynamic training...


In [ ]:
# ==============================================================================
#  DYNAMIC MODEL ANALYSIS - COMPREHENSIVE HEAD USAGE EVALUATION
# ==============================================================================
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import pandas as pd
import json
import time
import os

# =============================================================================
# CONFIGURATION
# =============================================================================
class Config:
    class_types = ['N', 'S', 'V', 'F', 'Q']
    num_heads = 8
    hidden_size = 2
    embedding = 16
    mlp_dim = 128
    kernel = 3
    emb_depth = 1
    batch_size = 128

    # Model weights paths
    baseline_weights = 'baseline_ecg_transformer.weights.h5'
    dynamic_weights = 'true_dynamic_ecg_transformer.weights.h5'

    # Data sources
    splits_file = 'consistent_data_splits.npz'
    baseline_results = 'baseline_results.json'
    dynamic_results = 'dynamic_results.json'

config = Config()

# =============================================================================
# MODEL COMPONENTS - DYNAMIC ARCHITECTURE
# =============================================================================
def generate_patch_conv_orgPaper_f(embedding, inputs):
    patches = layers.Conv2D(
        filters=embedding,
        kernel_size=[config.kernel, 1],
        strides=[config.kernel, 1],
        padding='valid'
    )(inputs)
    for _ in range(config.emb_depth - 1):
        patches = layers.BatchNormalization()(patches)
        patches = layers.Conv2D(
            filters=embedding,
            kernel_size=[config.kernel, 1],
            strides=[config.kernel, 1],
            padding='valid'
        )(patches)

    row_axis, col_axis = (1, 2)
    seq_len = (patches.shape[row_axis]) * (patches.shape[col_axis])
    x = layers.Reshape((seq_len, embedding))(patches)
    return x

class AddPositionEmbs(layers.Layer):
    def __init__(self, posemb_init=None, **kwargs):
        super().__init__(**kwargs)
        self.posemb_init = posemb_init

    def build(self, inputs_shape):
        pos_emb_shape = (1, inputs_shape[1], inputs_shape[2])
        self.pos_embedding = self.add_weight(
            name='pos_embedding',
            shape=pos_emb_shape,
            initializer=self.posemb_init,
            trainable=True
        )

    def call(self, inputs, inputs_positions=None):
        pos_embedding = tf.cast(self.pos_embedding, inputs.dtype)
        return inputs + pos_embedding

def mlp_block_f(mlp_dim, inputs):
    x = layers.Dense(units=mlp_dim, activation=tf.nn.gelu)(inputs)
    x = layers.Dropout(rate=0.1)(x)
    x = layers.Dense(units=inputs.shape[-1], activation=tf.nn.gelu)(x)
    x = layers.Dropout(rate=0.1)(x)
    return x

# Baseline model components
def BaselineEncoder1Dblock(num_heads, mlp_dim, inputs):
    x = layers.LayerNormalization(dtype=inputs.dtype)(inputs)
    x = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=config.hidden_size,
        dropout=0.1
    )(x, x)
    x = layers.Add()([x, inputs])
    y = layers.LayerNormalization(dtype=x.dtype)(x)
    y = mlp_block_f(mlp_dim, y)
    y_1 = layers.Add()([y, x])
    return y_1

def build_baseline_ViT():
    inputs = layers.Input(shape=(198, 1, 1), name='x1')
    RR_feat = layers.Input(shape=(2,), name='x2')

    patches = generate_patch_conv_orgPaper_f(config.embedding, inputs)
    x = AddPositionEmbs(
        posemb_init=tf.keras.initializers.RandomNormal(stddev=0.02)
    )(patches)
    x = layers.Dropout(rate=0.2)(x)
    x = BaselineEncoder1Dblock(config.num_heads, config.mlp_dim, x)
    encoded = layers.LayerNormalization(name='encoder_norm')(x)
    im_representation = layers.GlobalAveragePooling1D()(encoded)

    emb_RR = layers.Dense(units=2, use_bias=False)(RR_feat)
    im_representation = layers.Concatenate()([im_representation, emb_RR])
    logits = layers.Dense(
        units=len(config.class_types),
        name='head',
        kernel_initializer=tf.keras.initializers.zeros()
    )(im_representation)

    return tf.keras.Model(inputs=[inputs, RR_feat], outputs=logits)

# DYNAMIC MODEL COMPONENTS
class DynamicHeadGate(layers.Layer):
    """Pure dynamic gating - let the model decide how many heads it needs"""
    def __init__(self, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads

        # Gating network - learns to predict head importance
        self.gate_net = tf.keras.Sequential([
            layers.Dense(32, activation='relu'),
            layers.Dropout(0.1),
            layers.Dense(16, activation='relu'),
            layers.Dropout(0.1),
            layers.Dense(num_heads)  # Raw logits
        ])

        # Training step counter for temperature annealing
        self.training_step = tf.Variable(
            0, trainable=False, dtype=tf.int64, name="training_step"
        )

    def build(self, input_shape):
        super().build(input_shape)

    def call(self, inputs, training=None):
        signal_repr = tf.reduce_mean(inputs, axis=1)  # [batch_size, embed_dim]
        head_logits = self.gate_net(signal_repr, training=training)

        if training:
            self.training_step.assign_add(1)
            global_step = tf.cast(self.training_step, tf.float32)
            temperature = tf.maximum(0.1, 2.0 - global_step / 10000.0)

            uniform_noise = tf.random.uniform(
                tf.shape(head_logits), minval=1e-8, maxval=1.0
            )
            gumbel_noise = -tf.math.log(-tf.math.log(uniform_noise))
            gated_logits = (head_logits + gumbel_noise) / temperature
            head_probs = tf.nn.sigmoid(gated_logits)

            num_active = tf.reduce_sum(head_probs, axis=1)
            sparsity_penalty = tf.reduce_mean(
                tf.maximum(0.0, num_active - (self.num_heads * 0.8)) +
                tf.maximum(0.0, 1.0 - num_active)
            )
            self.add_loss(0.1 * sparsity_penalty)

            return head_probs
        else:
            head_probs = tf.nn.sigmoid(head_logits)
            threshold = 0.85
            head_mask = tf.cast(head_probs > threshold, tf.float32)

            num_active = tf.reduce_sum(head_mask, axis=1, keepdims=True)
            no_heads_active = tf.cast(num_active < 1.0, tf.float32)

            def activate_best_head():
                _, best_head_idx = tf.nn.top_k(head_probs, k=1)
                batch_size = tf.shape(head_probs)[0]
                batch_indices = tf.expand_dims(tf.range(batch_size), 1)
                indices = tf.stack([batch_indices, best_head_idx], axis=-1)
                best_head_mask = tf.scatter_nd(
                    indices,
                    tf.ones((batch_size, 1), dtype=tf.float32),
                    tf.shape(head_probs)
                )
                return tf.where(no_heads_active > 0, best_head_mask, head_mask)

            def keep_current_mask():
                return head_mask

            final_mask = tf.cond(
                tf.reduce_any(no_heads_active > 0),
                activate_best_head,
                keep_current_mask
            )
            return final_mask

class DynamicMultiHeadAttention(layers.Layer):
    def __init__(self, num_heads, key_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.dropout_rate = dropout

        # Individual attention heads
        self.attention_heads = []
        for i in range(num_heads):
            self.attention_heads.append(
                layers.MultiHeadAttention(
                    num_heads=1,
                    key_dim=key_dim,
                    dropout=dropout,
                    name=f'head_{i}'
                )
            )

        # Output projection
        self.output_dense = layers.Dense(key_dim * num_heads)
        self.dropout_layer = layers.Dropout(dropout)

        #  dynamic gating
        self.head_gate = DynamicHeadGate(num_heads)

        # Statistics tracking
        self.head_usage_counts = tf.Variable(
            initial_value=tf.zeros((num_heads,), dtype=tf.int64),
            trainable=False, name="head_usage_counts"
        )
        self.total_forward_passes = tf.Variable(
            0, trainable=False, dtype=tf.int64, name="total_forward_passes"
        )
        self.total_heads_computed = tf.Variable(
            0, trainable=False, dtype=tf.int64, name="total_heads_computed"
        )
        self.heads_per_sample = tf.Variable(
            initial_value=tf.zeros((10000,), dtype=tf.float32),
            trainable=False, name="heads_per_sample"
        )
        self.sample_idx = tf.Variable(
            0, trainable=False, dtype=tf.int64, name="sample_idx"
        )

        # Per-class head usage tracking
        self.class_head_usage = {}
        for i in range(len(config.class_types)):
            self.class_head_usage[i] = tf.Variable(
                initial_value=tf.zeros((num_heads,), dtype=tf.int64),
                trainable=False, name=f"class_{i}_head_usage"
            )
        self.class_counts = tf.Variable(
            initial_value=tf.zeros((len(config.class_types),), dtype=tf.int64),
            trainable=False, name="class_counts"
        )

    def build(self, input_shape):
        super().build(input_shape)

    def compute_output_shape(self, input_shape):
        if isinstance(input_shape, list):
            return input_shape[0]
        return input_shape

    def call(self, query, value=None, key=None, training=None, class_labels=None):
        if value is None:
            value = query
        if key is None:
            key = value

        batch_size = tf.shape(query)[0]
        head_mask = self.head_gate(query, training=training)

        head_outputs = []
        for i in range(self.num_heads):
            head_active = head_mask[:, i : i + 1]
            head_output = self.attention_heads[i](
                query, value, key, training=training
            )

            head_active_expanded = tf.expand_dims(head_active, axis=-1)
            gated_output = head_output * head_active_expanded
            head_outputs.append(gated_output)

        # Track statistics in eager mode
        if tf.executing_eagerly():
            self.total_forward_passes.assign_add(tf.cast(batch_size, tf.int64))
            batch_per_head_count = tf.cast(tf.reduce_sum(head_mask, axis=0), tf.int64)
            self.head_usage_counts.assign_add(batch_per_head_count)
            self.total_heads_computed.assign_add(tf.reduce_sum(batch_per_head_count))

            heads_used = tf.reduce_sum(head_mask, axis=1)
            for i in range(min(batch_size, 10)):
                idx = (self.sample_idx + tf.cast(i, tf.int64)) % 10000
                if idx < 10000:
                    self.heads_per_sample.scatter_nd_update([[idx]], [heads_used[i]])
            self.sample_idx.assign_add(tf.cast(batch_size, tf.int64))

        concatenated = tf.concat(head_outputs, axis=-1)
        output = self.output_dense(concatenated)
        output = self.dropout_layer(output, training=training)
        return output

    def get_comprehensive_stats(self):
        total_forward_passes = int(self.total_forward_passes.numpy())
        if total_forward_passes > 0:
            total_computed = int(self.total_heads_computed.numpy())
            total_possible = total_forward_passes * self.num_heads

            avg_heads = total_computed / total_forward_passes
            efficiency = 1.0 - (total_computed / total_possible)

            per_head_counts = [int(x) for x in self.head_usage_counts.numpy()]
            per_head_usage_rates = [
                (count / total_forward_passes) * 100.0 for count in per_head_counts
            ]

            heads_per_sample = self.heads_per_sample.numpy()
            valid_samples = heads_per_sample[heads_per_sample > 0]

            if len(valid_samples) > 0:
                min_heads = float(np.min(valid_samples))
                max_heads = float(np.max(valid_samples))
                std_heads = float(np.std(valid_samples))
                median_heads = float(np.median(valid_samples))

                head_distribution = {}
                for i in range(self.num_heads + 1):
                    count = np.sum(np.abs(valid_samples - i) < 0.1)
                    head_distribution[f'{i}_heads'] = int(count)

                percentiles = {}
                for p in [25, 50, 75, 90, 95]:
                    percentiles[f'p{p}'] = float(np.percentile(valid_samples, p))
            else:
                min_heads = max_heads = std_heads = median_heads = 0.0
                head_distribution = {}
                percentiles = {}

            return {
                'avg_heads_used': avg_heads,
                'min_heads_used': min_heads,
                'max_heads_used': max_heads,
                'median_heads_used': median_heads,
                'std_heads_used': std_heads,
                'total_heads': self.num_heads,
                'efficiency_gain': efficiency * 100.0,
                'total_computed': total_computed,
                'total_possible': total_possible,
                'total_forward_passes': total_forward_passes,
                'per_head_counts': per_head_counts,
                'per_head_usage_rates': per_head_usage_rates,
                'head_distribution': head_distribution,
                'percentiles': percentiles,
                'computational_savings': int(total_possible - total_computed),
                'samples_analyzed': int(len(valid_samples)),
            }
        return None

    def reset_stats(self):
        self.total_heads_computed.assign(0)
        self.total_forward_passes.assign(0)
        self.head_usage_counts.assign(tf.zeros_like(self.head_usage_counts))
        self.heads_per_sample.assign(tf.zeros_like(self.heads_per_sample))
        self.sample_idx.assign(0)

def TrueDynamicEncoder1Dblock(num_heads, mlp_dim, inputs):
    x = layers.LayerNormalization(dtype=inputs.dtype)(inputs)
    x_att = DynamicMultiHeadAttention(
        num_heads=num_heads,
        key_dim=config.hidden_size,
        dropout=0.1
    )(x, x)
    x = layers.Add()([x_att, inputs])

    y = layers.LayerNormalization(dtype=x.dtype)(x)
    y = mlp_block_f(mlp_dim, y)
    y_1 = layers.Add()([y, x])
    return y_1

def build_true_dynamic_ViT():
    inputs = layers.Input(shape=(198, 1, 1), name='x1')
    RR_feat = layers.Input(shape=(2,), name='x2')

    patches = generate_patch_conv_orgPaper_f(config.embedding, inputs)
    x = AddPositionEmbs(
        posemb_init=tf.keras.initializers.RandomNormal(stddev=0.02),
        name='posembed_input'
    )(patches)
    x = layers.Dropout(rate=0.1)(x)

    x = TrueDynamicEncoder1Dblock(config.num_heads, config.mlp_dim, x)

    encoded = layers.LayerNormalization(name='encoder_norm')(x)
    im_representation = layers.GlobalAveragePooling1D()(encoded)

    emb_RR = layers.Dense(units=4, activation='relu', use_bias=True)(RR_feat)
    emb_RR = layers.Dropout(0.1)(emb_RR)

    combined = layers.Concatenate()([im_representation, emb_RR])
    combined = layers.Dense(32, activation='relu')(combined)
    combined = layers.Dropout(0.2)(combined)

    logits = layers.Dense(
        units=len(config.class_types),
        name='head',
        kernel_initializer=tf.keras.initializers.TruncatedNormal(stddev=0.02)
    )(combined)

    model = tf.keras.Model(inputs=[inputs, RR_feat], outputs=logits)
    return model

# =============================================================================
# DATA LOADING
# =============================================================================
def load_test_data():
    """Load test data from consistent splits"""
    if not os.path.exists(config.splits_file):
        raise FileNotFoundError(
            f"Data splits file '{config.splits_file}' not found! Run baseline training first."
        )

    print(f"Loading test data from {config.splits_file}")
    data = np.load(config.splits_file)

    X_test = data['X_test']
    RR_test = data['RR_test']
    y_test = data['y_test']

    # Reshape and prepare
    X_test = X_test.reshape(-1, 198, 1, 1)
    y_test_categorical = tf.keras.utils.to_categorical(
        y_test, num_classes=len(config.class_types)
    )

    print(f"✅ Test data loaded: {len(y_test)} samples")
    return X_test, RR_test, y_test_categorical, y_test

def create_test_dataset(X_test, RR_test, y_test_categorical):
    """Create test dataset"""
    test_data = tf.data.Dataset.from_tensor_slices(X_test)
    test_RR = tf.data.Dataset.from_tensor_slices(RR_test)
    test_labels = tf.data.Dataset.from_tensor_slices(y_test_categorical)

    X_test_ds = tf.data.Dataset.zip((test_data, test_RR)).map(
        lambda x1, x2: {'x1': x1, 'x2': x2}
    )
    test_ds = tf.data.Dataset.zip((X_test_ds, test_labels))
    test_ds = test_ds.batch(config.batch_size).prefetch(tf.data.AUTOTUNE)
    return test_ds

# =============================================================================
# COMPREHENSIVE EVALUATION
# =============================================================================
def evaluate_model_comprehensive(model, test_ds, y_test, model_name):
    """Comprehensive model evaluation with detailed head analysis"""
    print(f"\n🧪 Comprehensive Evaluation: {model_name}")
    print("=" * 60)

    dynamic_attention_layer = None
    for layer in model.layers:
        if hasattr(layer, 'layers'):
            for sublayer in layer.layers:
                if isinstance(sublayer, DynamicMultiHeadAttention):
                    sublayer.reset_stats()
                    dynamic_attention_layer = sublayer
                    break
        elif isinstance(layer, DynamicMultiHeadAttention):
            layer.reset_stats()
            dynamic_attention_layer = layer
            break

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=2e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=['accuracy'],
        run_eagerly=True
    )

    start_time = time.time()

    logits_list = []
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    loss_fn = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

    print("Running inference and collecting head usage statistics...")
    for batch_idx, batch in enumerate(test_ds):
        inputs_dict, labels_onehot = batch

        logits = model(inputs_dict, training=False)
        logits_np = logits.numpy()
        logits_list.append(logits_np)

        bsize = logits_np.shape[0]
        total_samples += bsize

        batch_loss = loss_fn(labels_onehot, logits).numpy()
        total_loss += batch_loss * bsize

        preds_int = np.argmax(logits_np, axis=1)
        true_int = np.argmax(labels_onehot.numpy(), axis=1)
        total_correct += np.sum(preds_int == true_int)

        if (batch_idx + 1) % 20 == 0:
            print(f"  Processed {batch_idx + 1} batches... ({total_samples:,} samples)")

    inference_time = time.time() - start_time
    test_loss = total_loss / total_samples
    test_accuracy = total_correct / total_samples

    all_logits = np.concatenate(logits_list, axis=0)
    y_pred_probs = tf.nn.softmax(all_logits, axis=1).numpy()
    y_pred = np.argmax(y_pred_probs, axis=1)

    precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

    precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    print(f"\n📊 {model_name.upper()} EVALUATION RESULTS")
    print("=" * 60)
    print(f"🎯 Test Accuracy:      {test_accuracy:.6f} ({test_accuracy*100:.4f}%)")
    print(f"📉 Test Loss:          {test_loss:.6f}")
    print(f"⏱️  Inference Time:     {inference_time:.3f} seconds")
    print(f"🚀 Throughput:         {total_samples/inference_time:.1f} samples/sec")
    print(f"📊 Total Samples:      {total_samples:,}")

    print(f"\n📋 MACRO-AVERAGED METRICS:")
    print(f"   Precision: {precision_macro:.6f}")
    print(f"   Recall:    {recall_macro:.6f}")
    print(f"   F1-Score:  {f1_macro:.6f}")

    print(f"\n📋 WEIGHTED-AVERAGED METRICS:")
    print(f"   Precision: {precision_weighted:.6f}")
    print(f"   Recall:    {recall_weighted:.6f}")
    print(f"   F1-Score:  {f1_weighted:.6f}")

    print(f"\n📋 PER-CLASS PERFORMANCE:")
    per_class_precision = precision_score(y_test, y_pred, average=None, zero_division=0)
    per_class_recall = recall_score(y_test, y_pred, average=None, zero_division=0)
    per_class_f1 = f1_score(y_test, y_pred, average=None, zero_division=0)
    for i, class_name in enumerate(config.class_types):
        if i < len(per_class_precision):
            class_support = np.sum(y_test == i)
            print(
                f"   {class_name}: "
                f"Precision={per_class_precision[i]:.4f}, "
                f"Recall={per_class_recall[i]:.4f}, "
                f"F1={per_class_f1[i]:.4f} "
                f"(Support: {class_support:,})"
            )

    attention_stats = None
    if dynamic_attention_layer is not None:
        attention_stats = dynamic_attention_layer.get_comprehensive_stats()
        if attention_stats:
            print(f"\n🧠 DYNAMIC ATTENTION ANALYSIS")
            print("=" * 50)
            print(f"💡 EFFICIENCY SUMMARY:")
            print(f"   Average heads used:        {attention_stats['avg_heads_used']:.2f} / {attention_stats['total_heads']}")
            print(f"   Median heads used:         {attention_stats['median_heads_used']:.1f}")
            print(f"   Head usage range:          {attention_stats['min_heads_used']:.0f} - {attention_stats['max_heads_used']:.0f}")
            print(f"   Standard deviation:        {attention_stats['std_heads_used']:.2f}")
            print(f"   Computational savings:     {attention_stats['efficiency_gain']:.2f}%")

            print(f"\n📊 HEAD USAGE DISTRIBUTION:")
            total_analyzed = attention_stats['samples_analyzed']
            for k, v in sorted(attention_stats['head_distribution'].items()):
                if v > 0:
                    percentage = (v / total_analyzed) * 100 if total_analyzed > 0 else 0
                    print(f"   {k}: {v:,} samples ({percentage:.1f}%)")

            print(f"\n🔍 DETAILED HEAD STATISTICS:")
            for i, (count, rate) in enumerate(zip(
                attention_stats['per_head_counts'],
                attention_stats['per_head_usage_rates']
            )):
                print(f"   Head {i}: Used {count:,} times ({rate:.1f}% of samples)")

            print(f"\n📈 PERCENTILE ANALYSIS:")
            for p, value in attention_stats['percentiles'].items():
                print(f"   {p}: {value:.1f} heads")

    results = {
        'model_name': model_name,
        'test_accuracy': float(test_accuracy),
        'test_loss': float(test_loss),
        'precision_macro': float(precision_macro),
        'recall_macro': float(recall_macro),
        'f1_macro': float(f1_macro),
        'precision_weighted': float(precision_weighted),
        'recall_weighted': float(recall_weighted),
        'f1_weighted': float(f1_weighted),
        'per_class_precision': per_class_precision.tolist(),
        'per_class_recall': per_class_recall.tolist(),
        'per_class_f1': per_class_f1.tolist(),
        'inference_time': inference_time,
        'samples_per_second': total_samples / inference_time,
        'total_samples': total_samples,
        'predictions': y_pred.tolist(),
        'true_labels': y_test.tolist(),
        'attention_stats': attention_stats
    }
    return results

# =============================================================================
# COMPARISON AND VISUALIZATION
# =============================================================================
def compare_models(baseline_results, dynamic_results):
    """Comprehensive model comparison analysis"""
    print(f"\n{'='*80}")
    print(f"🔍 COMPREHENSIVE MODEL COMPARISON")
    print(f"{'='*80}")

    if isinstance(baseline_results, str):
        if os.path.exists(baseline_results):
            with open(baseline_results, 'r') as f:
                baseline_results = json.load(f)
        else:
            print(f"❌ Baseline results file not found: {baseline_results}")
            return

    baseline_acc = baseline_results['test_accuracy']
    dynamic_acc = dynamic_results['test_accuracy']
    acc_diff = dynamic_acc - baseline_acc
    acc_diff_percent = (acc_diff / baseline_acc) * 100

    baseline_f1 = baseline_results['f1_macro']
    dynamic_f1 = dynamic_results['f1_macro']
    f1_diff = dynamic_f1 - baseline_f1

    print(f"\n🎯 ACCURACY COMPARISON:")
    print(f"   Baseline Accuracy:     {baseline_acc:.6f} ({baseline_acc*100:.4f}%)")
    print(f"   Dynamic Accuracy:      {dynamic_acc:.6f} ({dynamic_acc*100:.4f}%)")
    print(f"   Absolute Difference:   {acc_diff:+.6f}")
    print(f"   Percentage Difference: {acc_diff*100:+.4f}%")
    print(f"   Relative Change:       {acc_diff_percent:+.4f}%")

    print(f"\n📊 F1-SCORE COMPARISON:")
    print(f"   Baseline F1 (macro):   {baseline_f1:.6f}")
    print(f"   Dynamic F1 (macro):    {dynamic_f1:.6f}")
    print(f"   F1 Difference:         {f1_diff:+.6f}")

    efficiency_gain = 0.0
    avg_heads_used = config.num_heads
    computational_savings = 0

    if dynamic_results['attention_stats']:
        stats = dynamic_results['attention_stats']
        efficiency_gain = stats['efficiency_gain']
        avg_heads_used = stats['avg_heads_used']
        computational_savings = stats['computational_savings']

        print(f"\n⚡ EFFICIENCY ANALYSIS:")
        print(f"   Average heads used:    {avg_heads_used:.2f} / {config.num_heads}")
        print(f"   Efficiency gain:       {efficiency_gain:.2f}%")
        print(f"   Operations saved:      {computational_savings:,}")
        print(f"   Head usage range:      {stats['min_heads_used']:.0f} - {stats['max_heads_used']:.0f}")

    baseline_time = baseline_results.get('inference_time', 0)
    dynamic_time = dynamic_results['inference_time']

    if baseline_time > 0:
        time_diff = dynamic_time - baseline_time
        speedup = baseline_time / dynamic_time

        print(f"\n⏱️ TIMING COMPARISON:")
        print(f"   Baseline inference:    {baseline_time:.3f} seconds")
        print(f"   Dynamic inference:     {dynamic_time:.3f} seconds")
        print(f"   Time difference:       {time_diff:+.3f} seconds")
        print(f"   Speedup factor:        {speedup:.2f}x")

    print(f"\n🏆 ANALYSIS VERDICT:")

    if abs(acc_diff) < 0.001:
        if efficiency_gain > 15:
            verdict = "🌟 EXCELLENT: Identical accuracy with significant efficiency gains!"
            recommendation = "Deploy dynamic model - it's strictly superior"
        elif efficiency_gain > 5:
            verdict = "✅ VERY GOOD: Same accuracy with moderate efficiency gains"
            recommendation = "Dynamic model is preferred"
        else:
            verdict = "🤔 INTERESTING: Same accuracy, minimal efficiency gains"
            recommendation = "Models are equivalent; dynamic adds complexity without clear benefit"
    elif acc_diff >= 0:
        verdict = "🌟 OUTSTANDING: Dynamic model improves both accuracy AND efficiency!"
        recommendation = "Definitely deploy the dynamic model"
    elif acc_diff >= -0.005 and efficiency_gain > 15:
        verdict = "✅ GOOD TRADE-OFF: Minor accuracy loss for substantial efficiency"
        recommendation = "Consider dynamic model for resource-constrained environments"
    elif acc_diff >= -0.01 and efficiency_gain > 25:
        verdict = "⚖️ ACCEPTABLE: Moderate accuracy loss for major efficiency gains"
        recommendation = "Evaluate based on specific deployment requirements"
    else:
        verdict = "❌ POOR TRADE-OFF: Accuracy loss too high for efficiency gains"
        recommendation = "Stick with baseline model or improve dynamic mechanism"

    print(f"   {verdict}")
    print(f"\n💡 RECOMMENDATION:")
    print(f"   {recommendation}")

    if abs(acc_diff) > 0.0001:
        efficiency_per_accuracy_loss = (
            efficiency_gain / abs(acc_diff) if acc_diff < 0 else float('inf')
        )
        print(f"\n📈 COST-BENEFIT METRICS:")
        if acc_diff < 0:
            print(
                f"   Efficiency gain per 0.001 accuracy loss: "
                f"{efficiency_per_accuracy_loss*0.001:.1f}%"
            )
        else:
            print(f"   Efficiency gained with accuracy improvement: {efficiency_gain:.1f}%")

    return {
        'accuracy_difference': acc_diff,
        'f1_difference': f1_diff,
        'efficiency_gain': efficiency_gain,
        'avg_heads_used': avg_heads_used,
        'verdict': verdict,
        'recommendation': recommendation
    }

def plot_comprehensive_analysis(baseline_results, dynamic_results, save_path='comprehensive_analysis.png'):
    """Create comprehensive visualization of the analysis"""
    if isinstance(baseline_results, str):
        with open(baseline_results, 'r') as f:
            baseline_results = json.load(f)

    fig, axes = plt.subplots(3, 3, figsize=(20, 15))
    fig.suptitle('Comprehensive Dynamic Attention Analysis', fontsize=16, fontweight='bold')

    # 1. Accuracy Comparison (Exact values)
    models = ['Baseline', 'True Dynamic']
    accuracies = [baseline_results['test_accuracy'], dynamic_results['test_accuracy']]

    bars = axes[0, 0].bar(models, accuracies, color=['skyblue', 'lightgreen'], alpha=0.8)
    axes[0, 0].set_title('Exact Accuracy Comparison')
    axes[0, 0].set_ylabel('Accuracy')
    min_acc = min(accuracies)
    max_acc = max(accuracies)
    axes[0, 0].set_ylim([min_acc - 0.002, max_acc + 0.002])
    axes[0, 0].grid(True, alpha=0.3)

    for bar, acc in zip(bars, accuracies):
        axes[0, 0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.0005,
            f'{acc:.6f}',
            ha='center',
            va='bottom',
            fontweight='bold',
            fontsize=10
        )

    # 2. Per-class F1 comparison
    class_names = config.class_types
    baseline_f1s = baseline_results['per_class_f1'][:len(class_names)]
    dynamic_f1s = dynamic_results['per_class_f1'][:len(class_names)]

    x = np.arange(len(class_names))
    width = 0.35

    axes[0, 1].bar(x - width / 2, baseline_f1s, width, label='Baseline', alpha=0.8, color='skyblue')
    axes[0, 1].bar(x + width / 2, dynamic_f1s, width, label='True Dynamic', alpha=0.8, color='lightgreen')
    axes[0, 1].set_title('Per-Class F1-Score Comparison')
    axes[0, 1].set_ylabel('F1-Score')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels(class_names)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Overall metrics comparison
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    baseline_vals = [
        baseline_results['test_accuracy'],
        baseline_results['precision_macro'],
        baseline_results['recall_macro'],
        baseline_results['f1_macro']
    ]
    dynamic_vals = [
        dynamic_results['test_accuracy'],
        dynamic_results['precision_macro'],
        dynamic_results['recall_macro'],
        dynamic_results['f1_macro']
    ]

    x = np.arange(len(metrics))
    axes[0, 2].bar(x - width / 2, baseline_vals, width, label='Baseline', alpha=0.8, color='skyblue')
    axes[0, 2].bar(x + width / 2, dynamic_vals, width, label='True Dynamic', alpha=0.8, color='lightgreen')
    axes[0, 2].set_title('Overall Metrics Comparison')
    axes[0, 2].set_ylabel('Score')
    axes[0, 2].set_xticks(x)
    axes[0, 2].set_xticklabels(metrics, rotation=45)
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)

    # 4. Head usage rates
    if dynamic_results['attention_stats']:
        stats = dynamic_results['attention_stats']
        head_usage = stats['per_head_usage_rates']
        head_labels = [f'H{i}' for i in range(len(head_usage))]

        bars = axes[1, 0].bar(head_labels, head_usage, color='orange', alpha=0.7)
        axes[1, 0].set_title('Head Usage Rates (True Dynamic)')
        axes[1, 0].set_ylabel('Usage Rate (%)')
        axes[1, 0].grid(True, alpha=0.3)

        for bar, rate in zip(bars, head_usage):
            axes[1, 0].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 1,
                f'{rate:.1f}%',
                ha='center',
                va='bottom',
                fontsize=8
            )
    else:
        axes[1, 0].text(
            0.5, 0.5, 'No head usage\ndata available',
            ha='center', va='center', transform=axes[1, 0].transAxes, fontsize=12
        )
        axes[1, 0].set_title('Head Usage Rates')

    # 5. Head distribution histogram
    if dynamic_results['attention_stats'] and 'head_distribution' in dynamic_results['attention_stats']:
        dist = dynamic_results['attention_stats']['head_distribution']
        heads_used = []
        sample_counts = []

        for k, v in sorted(dist.items()):
            if v > 0:
                heads_used.append(int(k.replace('_heads', '')))
                sample_counts.append(v)

        if heads_used:
            bars = axes[1, 1].bar(heads_used, sample_counts, alpha=0.7, color='green')
            axes[1, 1].set_title('Distribution of Heads per Sample')
            axes[1, 1].set_xlabel('Number of Heads Used')
            axes[1, 1].set_ylabel('Number of Samples')
            axes[1, 1].grid(True, alpha=0.3)

            for bar, count in zip(bars, sample_counts):
                axes[1, 1].text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(sample_counts) * 0.01,
                    f'{count:,}',
                    ha='center',
                    va='bottom',
                    fontsize=8
                )
        else:
            axes[1, 1].text(
                0.5, 0.5, 'No distribution\ndata available',
                ha='center', va='center', transform=axes[1, 1].transAxes
            )
    else:
        axes[1, 1].text(
            0.5, 0.5, 'No distribution\ndata available',
            ha='center', va='center', transform=axes[1, 1].transAxes
        )
        axes[1, 1].set_title('Head Distribution')

    # 6. Efficiency metrics
    if dynamic_results['attention_stats']:
        stats = dynamic_results['attention_stats']
        efficiency_metrics = ['Avg Heads Used', 'Efficiency Gain (%)', 'Computational Savings (k)']
        efficiency_values = [
            stats['avg_heads_used'],
            stats['efficiency_gain'],
            stats['computational_savings'] / 1000
        ]

        bars = axes[1, 2].bar(
            efficiency_metrics,
            efficiency_values,
            color=['blue', 'red', 'purple'],
            alpha=0.7
        )
        axes[1, 2].set_title('Efficiency Metrics')
        axes[1, 2].set_ylabel('Value')
        axes[1, 2].tick_params(axis='x', rotation=45)
        axes[1, 2].grid(True, alpha=0.3)

        for bar, val in zip(bars, efficiency_values):
            axes[1, 2].text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(efficiency_values) * 0.01,
                f'{val:.1f}',
                ha='center',
                va='bottom',
                fontweight='bold'
            )
    else:
        axes[1, 2].text(
            0.5, 0.5, 'No efficiency\ndata available',
            ha='center', va='center', transform=axes[1, 2].transAxes
        )

    # 7. Accuracy difference visualization
    acc_diff = dynamic_results['test_accuracy'] - baseline_results['test_accuracy']
    f1_diff = dynamic_results['f1_macro'] - baseline_results['f1_macro']

    diffs = ['Accuracy Diff', 'F1 Diff (macro)']
    diff_vals = [acc_diff * 100, f1_diff * 100]
    colors = ['green' if x >= 0 else 'red' for x in diff_vals]

    bars = axes[2, 0].bar(diffs, diff_vals, color=colors, alpha=0.7)
    axes[2, 0].set_title('Performance Differences (Percentage Points)')
    axes[2, 0].set_ylabel('Difference (%)')
    axes[2, 0].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[2, 0].grid(True, alpha=0.3)

    for bar, val in zip(bars, diff_vals):
        axes[2, 0].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (0.01 if val >= 0 else -0.01),
            f'{val:+.4f}%',
            ha='center',
            va='bottom' if val >= 0 else 'top',
            fontweight='bold'
        )

    # 8. Confusion matrix comparison (simplified)
    cm_baseline = confusion_matrix(baseline_results['true_labels'], baseline_results['predictions'])
    im1 = axes[2, 1].imshow(cm_baseline, interpolation='nearest', cmap='Blues')
    axes[2, 1].set_title('Baseline Confusion Matrix')
    axes[2, 1].set_xticks(range(len(config.class_types)))
    axes[2, 1].set_yticks(range(len(config.class_types)))
    axes[2, 1].set_xticklabels(config.class_types)
    axes[2, 1].set_yticklabels(config.class_types)

    cm_dynamic = confusion_matrix(dynamic_results['true_labels'], dynamic_results['predictions'])
    im2 = axes[2, 2].imshow(cm_dynamic, interpolation='nearest', cmap='Greens')
    axes[2, 2].set_title('True Dynamic Confusion Matrix')
    axes[2, 2].set_xticks(range(len(config.class_types)))
    axes[2, 2].set_yticks(range(len(config.class_types)))
    axes[2, 2].set_xticklabels(config.class_types)
    axes[2, 2].set_yticklabels(config.class_types)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

def create_detailed_report(baseline_results, dynamic_results, comparison_results):
    """Create a detailed text report of the analysis"""
    report = []
    report.append("=" * 80)
    report.append("COMPREHENSIVE TRUE DYNAMIC ATTENTION ANALYSIS REPORT")
    report.append("=" * 80)
    report.append("")

    # Executive Summary
    report.append("📋 EXECUTIVE SUMMARY")
    report.append("-" * 40)
    report.append(
        f"Baseline Model Accuracy:    {baseline_results['test_accuracy']:.6f} "
        f"({baseline_results['test_accuracy']*100:.4f}%)"
    )
    report.append(
        f"True Dynamic Accuracy:      {dynamic_results['test_accuracy']:.6f} "
        f"({dynamic_results['test_accuracy']*100:.4f}%)"
    )
    report.append(
        f"Accuracy Change:            "
        f"{comparison_results['accuracy_difference']:+.6f} "
        f"({comparison_results['accuracy_difference']*100:+.4f}%)"
    )

    if dynamic_results['attention_stats']:
        stats = dynamic_results['attention_stats']
        report.append(f"Average Heads Used:         {stats['avg_heads_used']:.2f} / {config.num_heads}")
        report.append(f"Computational Savings:      {stats['efficiency_gain']:.2f}%")
        report.append(f"Operations Saved:           {stats['computational_savings']:,}")

    report.append("")
    report.append(f"🏆 VERDICT: {comparison_results['verdict']}")
    report.append(f"💡 RECOMMENDATION: {comparison_results['recommendation']}")
    report.append("")

    # Detailed Performance Analysis
    report.append("📊 DETAILED PERFORMANCE ANALYSIS")
    report.append("-" * 40)

    metrics = [
        ('Test Accuracy', 'test_accuracy'),
        ('Test Loss', 'test_loss'),
        ('Precision (macro)', 'precision_macro'),
        ('Recall (macro)', 'recall_macro'),
        ('F1-Score (macro)', 'f1_macro'),
        ('Precision (weighted)', 'precision_weighted'),
        ('Recall (weighted)', 'recall_weighted'),
        ('F1-Score (weighted)', 'f1_weighted')
    ]

    for metric_name, metric_key in metrics:
        baseline_val = baseline_results[metric_key]
        dynamic_val = dynamic_results[metric_key]
        diff = dynamic_val - baseline_val
        report.append(
            f"{metric_name:<20} | Baseline: {baseline_val:.6f} | "
            f"Dynamic: {dynamic_val:.6f} | Diff: {diff:+.6f}"
        )

    report.append("")

    # Per-class analysis
    report.append("📋 PER-CLASS PERFORMANCE ANALYSIS")
    report.append("-" * 40)
    report.append(f"{'Class':<8} | {'Baseline F1':<12} | {'Dynamic F1':<12} | {'Difference':<12} | {'Support':<10}")
    report.append("-" * 65)

    for i, class_name in enumerate(config.class_types):
        if i < len(baseline_results['per_class_f1']) and i < len(dynamic_results['per_class_f1']):
            baseline_f1 = baseline_results['per_class_f1'][i]
            dynamic_f1 = dynamic_results['per_class_f1'][i]
            f1_diff = dynamic_f1 - baseline_f1
            support = np.sum(np.array(dynamic_results['true_labels']) == i)
            report.append(
                f"{class_name:<8} | {baseline_f1:<12.6f} | {dynamic_f1:<12.6f} | "
                f"{f1_diff:+12.6f} | {support:<10,}"
            )

    report.append("")

    # Head usage analysis
    if dynamic_results['attention_stats']:
        stats = dynamic_results['attention_stats']
        report.append("🧠 DYNAMIC ATTENTION HEAD USAGE ANALYSIS")
        report.append("-" * 40)
        report.append(f"Average Heads Used:         {stats['avg_heads_used']:.2f} / {stats['total_heads']}")
        report.append(f"Median Heads Used:          {stats['median_heads_used']:.1f}")
        report.append(f"Head Usage Range:           {stats['min_heads_used']:.0f} - {stats['max_heads_used']:.0f}")
        report.append(f"Standard Deviation:         {stats['std_heads_used']:.2f}")
        report.append(f"Efficiency Gain:            {stats['efficiency_gain']:.2f}%")
        report.append(f"Computational Savings:      {stats['computational_savings']:,} operations")
        report.append("")

        report.append("Per-Head Usage Statistics:")
        for i, (count, rate) in enumerate(zip(
            stats['per_head_counts'],
            stats['per_head_usage_rates']
        )):
            report.append(f"  Head {i}: {count:,} uses ({rate:.1f}% of samples)")
        report.append("")

        report.append("Head Usage Distribution:")
        total_samples = stats['samples_analyzed']
        for k, v in sorted(stats['head_distribution'].items()):
            if v > 0:
                percentage = (v / total_samples) * 100 if total_samples > 0 else 0
                report.append(f"  {k}: {v:,} samples ({percentage:.1f}%)")
        report.append("")

    report_text = "\n".join(report)
    with open('detailed_analysis_report.txt', 'w') as f:
        f.write(report_text)

    print(f"📄 Detailed report saved to: detailed_analysis_report.txt")
    return report_text

# =============================================================================
# MAIN EXECUTION
# =============================================================================
def main():
    """Main execution for comprehensive true dynamic analysis"""
    print("🔍 TRUE DYNAMIC MODEL - COMPREHENSIVE HEAD USAGE ANALYSIS")
    print("=" * 70)

    try:
        gpus = tf.config.experimental.list_physical_devices('GPU')
        if gpus:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"🚀 Using {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"⚠️ GPU setup warning: {e}")

    required_files = [config.baseline_weights, config.dynamic_weights, config.splits_file]
    missing_files = [f for f in required_files if not os.path.exists(f)]

    if missing_files:
        print(f"❌ Missing required files:")
        for f in missing_files:
            print(f"   - {f}")
        print(f"\n💡 Make sure you've run both baseline and dynamic training scripts first!")
        return

    try:
        X_test, RR_test, y_test_categorical, y_test = load_test_data()
        test_ds = create_test_dataset(X_test, RR_test, y_test_categorical)

        unique, counts = np.unique(y_test, return_counts=True)
        print(f"\n📊 Test Data Information:")
        print(f"   Total samples: {len(y_test):,}")
        for i, (class_idx, count) in enumerate(zip(unique, counts)):
            if class_idx < len(config.class_types):
                print(f"   {config.class_types[class_idx]}: {count:,} ({count/len(y_test)*100:.1f}%)")
    except Exception as e:
        print(f"❌ Failed to load test data: {e}")
        return

    print(f"\n🏗️ Evaluating Baseline Model...")
    try:
        if os.path.exists(config.baseline_results):
            print(f"📂 Loading baseline results from {config.baseline_results}")
            with open(config.baseline_results, 'r') as f:
                baseline_results = json.load(f)
        else:
            baseline_model = build_baseline_ViT()
            baseline_model.load_weights(config.baseline_weights)
            baseline_results = evaluate_model_comprehensive(
                baseline_model, test_ds, y_test, "Baseline"
            )
            with open(config.baseline_results, 'w') as f:
                json.dump(
                    {k: v for k, v in baseline_results.items() if k not in ['predictions', 'true_labels']},
                    f, indent=2
                )
    except Exception as e:
        print(f"❌ Failed to load/evaluate baseline model: {e}")
        return

    print(f"\n🏗️ Evaluating True Dynamic Model...")
    try:
        dynamic_model = build_true_dynamic_ViT()

        # ── Step 1: Build every layer’s variables via a dummy pass ──
        dummy_ecg = tf.zeros((1, 198, 1, 1), dtype=tf.float32)
        dummy_rr  = tf.zeros((1,   2    ), dtype=tf.float32)
        _ = dynamic_model([dummy_ecg, dummy_rr], training=False)

        # ── Step 2: Load weights (by order) ──
        dynamic_model.load_weights(config.dynamic_weights)
        print("✅ Loaded dynamic model weights.")

        dynamic_results = evaluate_model_comprehensive(
            dynamic_model, test_ds, y_test, "True Dynamic"
        )
        with open(config.dynamic_results, 'w') as f:
            json.dump(
                {k: v for k, v in dynamic_results.items() if k not in ['predictions', 'true_labels']},
                f, indent=2
            )
    except Exception as e:
        print(f"❌ Failed to load/evaluate true dynamic model: {e}")
        return

    print(f"\n🔍 Performing Comprehensive Comparison...")
    comparison_results = compare_models(baseline_results, dynamic_results)

    print(f"\n📊 Generating Comprehensive Visualizations...")
    try:
        plot_comprehensive_analysis(baseline_results, dynamic_results)
        print(f"✅ Visualizations saved to: comprehensive_analysis.png")
    except Exception as e:
        print(f"⚠️ Warning: Could not create visualizations: {e}")

    print(f"\n📄 Creating Detailed Report...")
    try:
        create_detailed_report(baseline_results, dynamic_results, comparison_results)
    except Exception as e:
        print(f"⚠️ Warning: Could not create detailed report: {e}")

    print(f"\n🎉 ANALYSIS COMPLETE!")
    print(f"📁 Generated Files:")
    print(f"   - comprehensive_analysis.png (visualizations)")
    print(f"   - detailed_analysis_report.txt (comprehensive report)")
    print(f"   - {config.baseline_results} (baseline metrics)")
    print(f"   - {config.dynamic_results} (dynamic metrics)")

    return baseline_results, dynamic_results, comparison_results

if __name__ == "__main__":
    baseline_results, dynamic_results, comparison_results = main()
